# TAREA 10

Vimos en clase el concepto de minimizar la varianza conjunta para realizar particiones en árboles de regresión; dado que en clasificación las salidas son categorías, no se puede usar el mismo concepto.

Investiga los siguientes conceptos y su conexión con las particiones en árboles de clasificación:



##  GINI (Índice de Gini / Gini Impurity)

Mide la probabilidad de **clasificar incorrectamente** una muestra elegida al azar si se le asigna una clase según la distribución del nodo.

### Fórmula

$$\text{Gini} = 1 - \sum_{i=1}^{k} p_i^2$$

donde $p_i$ es la proporción de muestras de la clase $i$ en el nodo.

### Interpretación

- **Gini = 0** → nodo completamente puro (una sola clase) ✅
- **Gini = 0.5** (caso binario) → máxima impureza, clases al 50/50 ❌
- El valor máximo es $1 - \frac{1}{k}$ para $k$ clases

### Conexión con particiones

En cada nodo se calcula el **Gini ponderado** del split:

$$\text{Gini}_{split} = \sum_{j} \frac{n_j}{n} \cdot \text{Gini}(j)$$

Se elige el atributo que **minimiza** el Gini ponderado (equivale a **maximizar** la reducción de impureza).

$$\Delta\text{Gini} = \text{Gini}_{raíz} - \text{Gini}_{split}$$


##  Entropía (Information Gain)

Mide el **desorden o incertidumbre** dentro de un conjunto de datos. Viene de la teoría de la información de Shannon.

### Fórmula

$$H = -\sum_{i=1}^{k} p_i \log_2(p_i)$$

### Interpretación

- **H = 0** → nodo puro, certeza total ✅
- **H = 1** (caso binario) → máxima incertidumbre ❌
- El valor máximo es $\log_2(k)$ para $k$ clases

### Conexión con particiones

Se calcula la **Ganancia de Información (IG)**: cuánta incertidumbre se elimina al hacer el split.

$$IG = H_{raíz} - \sum_{j} \frac{n_j}{n} \cdot H(j)$$

Se elige el atributo que **maximiza** el IG. Este criterio es usado por los algoritmos **ID3** y **C4.5**.

##  Log Loss (Pérdida Logarítmica / Cross-Entropy Loss)

Mide **qué tan bien calibradas** están las probabilidades predichas por el modelo respecto a las etiquetas reales.

### Fórmula (clasificación binaria)

$$\text{Log Loss} = -\frac{1}{n}\sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i) \right]$$

donde $\hat{p}_i$ es la probabilidad predicha de clase positiva para la muestra $i$.

### Interpretación

- **Log Loss = 0** → predicciones perfectas ✅
- **Log Loss alto** → predicciones muy alejadas de la realidad ❌
- Penaliza fuertemente las predicciones **confiadas pero incorrectas**

### Conexión con particiones

A diferencia de Gini y Entropía, el Log Loss **no se usa para elegir splits** en árboles clásicos (CART, ID3). Su rol es:

1. **Evaluar** qué tan bien predice el modelo en las hojas finales
2. **Función de pérdida** en modelos como Gradient Boosting (XGBoost, LightGBM)

En una hoja, el árbol asigna $\hat{p} = \frac{\text{muestras Yes}}{\text{total en hoja}}$ y el Log Loss mide qué tan lejos está esa probabilidad de las etiquetas reales.

##  ¿Cuál es la diferencia entre Entropía y Log Loss?



### Comparación directa

| Aspecto | Entropía | Log Loss |
|---|---|---|
| **¿Qué mide?** | Impureza de un nodo | Error de predicción probabilística |
| **¿Cuándo se usa?** | Durante la construcción del árbol (elegir splits) | Para evaluar o entrenar el modelo completo |
| **Perspectiva** | Propiedad de los *datos en el nodo* | Distancia entre predicciones y etiquetas reales |
| **¿Sobre qué $p$?** | Frecuencia observada de clases en el nodo | Probabilidad *predicha* por el modelo |
| **Algoritmos** | ID3, C4.5 | XGBoost, redes neuronales, evaluación |

### La conexión matemática

$$\text{Entropía}: \quad H = -\sum p_i \log p_i \quad \leftarrow p_i \text{ es la frecuencia real en el nodo}$$

$$\text{Log Loss}: \quad LL = -\frac{1}{n}\sum y_i \log \hat{p}_i \quad \leftarrow \hat{p}_i \text{ es la probabilidad predicha por el modelo}$$

> La **entropía** es un caso especial de Log Loss donde la probabilidad predicha coincide exactamente con la frecuencia observada. En otras palabras, la entropía es el Log Loss **mínimo posible** que podría tener ese nodo.

### Resumen en una línea

- **Entropía** → mide la incertidumbre de los *datos* para decidir dónde partir
- **Log Loss** → mide qué tan mal *predice el modelo* esas probabilidades

# Escoge un dataset y realiza el ejemplo de hacer una partición para cada uno de los criterios de decisión.



In [ ]:
import pandas as pd
import numpy as np


In [ ]:

df = pd.read_csv("/content/play_tennis.csv")
print("=== Dataset Play Tennis ===")
print(df.to_string(index=True))
print(f"\nTotal: {len(df)} muestras | Yes: {(df.play=='Yes').sum()} | No: {(df.play=='No').sum()}")

=== Dataset Play Tennis ===
    day   outlook  temp humidity    wind play
0    D1     Sunny   Hot     High    Weak   No
1    D2     Sunny   Hot     High  Strong   No
2    D3  Overcast   Hot     High    Weak  Yes
3    D4      Rain  Mild     High    Weak  Yes
4    D5      Rain  Cool   Normal    Weak  Yes
5    D6      Rain  Cool   Normal  Strong   No
6    D7  Overcast  Cool   Normal  Strong  Yes
7    D8     Sunny  Mild     High    Weak   No
8    D9     Sunny  Cool   Normal    Weak  Yes
9   D10      Rain  Mild   Normal    Weak  Yes
10  D11     Sunny  Mild   Normal  Strong  Yes
11  D12  Overcast  Mild     High  Strong  Yes
12  D13  Overcast   Hot   Normal    Weak  Yes
13  D14      Rain  Mild     High  Strong   No

Total: 14 muestras | Yes: 9 | No: 5


In [ ]:
print("=== Distribución por Outlook ===\n")
for outlook in df['outlook'].unique():
    subset = df[df.outlook == outlook]
    yes = (subset.play == 'Yes').sum()
    no  = (subset.play == 'No').sum()
    print(f"{outlook:10s} → {len(subset)} muestras | Yes: {yes} | No: {no}")

=== Distribución por Outlook ===

Sunny      → 5 muestras | Yes: 2 | No: 3
Overcast   → 4 muestras | Yes: 4 | No: 0
Rain       → 5 muestras | Yes: 3 | No: 2


In [ ]:
def gini(subset):
    n = len(subset)
    if n == 0:
        return 0
    p_yes = (subset.play == 'Yes').sum() / n
    p_no  = (subset.play == 'No').sum()  / n
    return 1 - p_yes**2 - p_no**2

print("=== GINI ===\n")

g_raiz = gini(df)
print(f"Gini(raíz) = {g_raiz:.4f}\n")

gini_ponderado = 0
for nombre, subset in df.groupby('outlook'):
    g = gini(subset)
    peso = len(subset) / len(df)
    gini_ponderado += peso * g
    print(f"Gini({nombre:10s}) = {g:.4f}  ({len(subset)}/{len(df)} muestras)")

print(f"\nGini ponderado(outlook) = {gini_ponderado:.4f}")
print(f"Reducción de impureza   = {g_raiz:.4f} - {gini_ponderado:.4f} = {g_raiz - gini_ponderado:.4f}")

=== GINI ===

Gini(raíz) = 0.4592

Gini(Overcast  ) = 0.0000  (4/14 muestras)
Gini(Rain      ) = 0.4800  (5/14 muestras)
Gini(Sunny     ) = 0.4800  (5/14 muestras)

Gini ponderado(outlook) = 0.3429
Reducción de impureza   = 0.4592 - 0.3429 = 0.1163


In [ ]:
def entropia(subset):
    n = len(subset)
    if n == 0:
        return 0
    h = 0
    for clase in ['Yes', 'No']:
        p = (subset.play == clase).sum() / n
        if p > 0:
            h -= p * np.log2(p)
    return h

print("=== ENTROPÍA e INFORMATION GAIN ===\n")

h_raiz = entropia(df)
print(f"H(raíz) = {h_raiz:.4f} bits\n")

h_ponderada = 0
for nombre, subset in df.groupby('outlook'):
    h = entropia(subset)
    peso = len(subset) / len(df)
    h_ponderada += peso * h
    print(f"H({nombre:10s}) = {h:.4f} bits  ({len(subset)}/{len(df)} muestras)")

ig = h_raiz - h_ponderada
print(f"\nH ponderada(outlook)  = {h_ponderada:.4f} bits")
print(f"Information Gain (IG) = {h_raiz:.4f} - {h_ponderada:.4f} = {ig:.4f} bits")

=== ENTROPÍA e INFORMATION GAIN ===

H(raíz) = 0.9403 bits

H(Overcast  ) = 0.0000 bits  (4/14 muestras)
H(Rain      ) = 0.9710 bits  (5/14 muestras)
H(Sunny     ) = 0.9710 bits  (5/14 muestras)

H ponderada(outlook)  = 0.6935 bits
Information Gain (IG) = 0.9403 - 0.6935 = 0.2467 bits


In [ ]:
def log_loss_nodo(subset):
    n = len(subset)
    if n == 0:
        return 0
    p_hat = (subset.play == 'Yes').sum() / n
    p = np.clip(p_hat, 1e-10, 1 - 1e-10)
    loss = 0
    for _, row in subset.iterrows():
        y = 1 if row.play == 'Yes' else 0
        loss += y * np.log(p) + (1 - y) * np.log(1 - p)
    return -loss / n

print("=== LOG LOSS ===\n")

ll_ponderado = 0
for nombre, subset in df.groupby('outlook'):
    p_hat = (subset.play == 'Yes').sum() / len(subset)
    ll = log_loss_nodo(subset)
    peso = len(subset) / len(df)
    ll_ponderado += peso * ll
    print(f"Hoja {nombre:10s} → p̂(Yes)={p_hat:.2f} | LL={ll:.4f}  ({len(subset)}/{len(df)} muestras)")

print(f"\nLog Loss global (outlook) = {ll_ponderado:.4f}")

=== LOG LOSS ===

Hoja Overcast   → p̂(Yes)=1.00 | LL=0.0000  (4/14 muestras)
Hoja Rain       → p̂(Yes)=0.60 | LL=0.6730  (5/14 muestras)
Hoja Sunny      → p̂(Yes)=0.40 | LL=0.6730  (5/14 muestras)

Log Loss global (outlook) = 0.4807


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import LabelEncoder

print("=== RESUMEN COMPARATIVO ===\n")
print(f"{'Métrica':<12} {'Raíz':>10} {'Tras split':>12} {'Mejora':>10}")
print("-" * 48)
print(f"{'Gini':<12} {g_raiz:>10.4f} {gini_ponderado:>12.4f} {g_raiz - gini_ponderado:>10.4f}")
print(f"{'Entropía':<12} {h_raiz:>10.4f} {h_ponderada:>12.4f} {ig:>10.4f}")
print(f"{'Log Loss':<12} {'---':>10} {ll_ponderado:>12.4f} {'↓ mejor':>10}")

print("\n=== ÁRBOL con sklearn (criterio=gini) ===\n")

df_enc = df.copy()
for col in ['outlook', 'temp', 'humidity', 'wind', 'play']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

X = df_enc[['outlook', 'temp', 'humidity', 'wind']]
y = df_enc['play']

clf = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
clf.fit(X, y)
print(export_text(clf, feature_names=['outlook', 'temp', 'humidity', 'wind']))

=== RESUMEN COMPARATIVO ===

Métrica            Raíz   Tras split     Mejora
------------------------------------------------
Gini             0.4592       0.3429     0.1163
Entropía         0.9403       0.6935     0.2467
Log Loss            ---       0.4807    ↓ mejor

=== ÁRBOL con sklearn (criterio=gini) ===

|--- outlook <= 0.50
|   |--- class: 1
|--- outlook >  0.50
|   |--- humidity <= 0.50
|   |   |--- outlook <= 1.50
|   |   |   |--- class: 0
|   |   |--- outlook >  1.50
|   |   |   |--- class: 0
|   |--- humidity >  0.50
|   |   |--- wind <= 0.50
|   |   |   |--- class: 0
|   |   |--- wind >  0.50
|   |   |   |--- class: 1

